[Reference](https://medium.com/@Rohan_Dutt/10-forecasting-techniques-essential-for-optimizing-inventory-in-fast-fashion-and-e-commerce-2d1afa6a4fec$0)

# 1. Gradient Boosted Decision Trees (XGBoost & LightGBM)


In [2]:
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

# Assume df contains: 'sales', 'price', 'is_promo', 'temperature', 'day_of_week'
# Target variable is shifted to predict next week's sales
X = df[['price', 'is_promo', 'temperature', 'day_of_week']]
y = df['sales_next_week']
# Advanced parameters for robust e-commerce forecasting
params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.05,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'early_stopping_rounds': 10
}
# Time-series cross-validation to prevent data leakage
tscv = TimeSeriesSplit(n_splits=5)
for train_index, test_index in tscv.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

preds = model.predict(X_test)
print(f"RMSE: {mean_squared_error(y_test, preds, squared=False):.2f}")

# 2. Temporal Fusion Transformers (TFT)


In [3]:
import pandas as pd
import pytorch_lightning as pl
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer

# Create dataset mapping static and dynamic e-commerce features
dataset = TimeSeriesDataSet(
    df,
    time_idx="time_step",
    target="sales",
    group_ids=["sku_id", "store_id"],
    min_encoder_length=24,  # Look back 24 weeks
    max_prediction_length=12, # Predict 12 weeks out
    static_categoricals=["category", "brand"],
    time_varying_known_categoricals=["month", "is_holiday"],
    time_varying_unknown_reals=["sales", "page_views"]
)
dataloader = dataset.to_dataloader(train=True, batch_size=64)
# Initialize TFT Architecture
tft = TemporalFusionTransformer.from_dataset(
    dataset,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=1,
    dropout=0.1,
    hidden_continuous_size=8,
    output_size=7,  # Quantile forecasts (e.g., 10th to 90th percentile)
    loss=pl.metrics.QuantileLoss()
)
trainer = pl.Trainer(max_epochs=10, accelerator="gpu")
trainer.fit(tft, train_dataloaders=dataloader)

# 3. Transfer Learning for New Product Introductions (NPI)


In [4]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# Load pre-trained ResNet (Feature Extractor)
weights = ResNet50_Weights.DEFAULT
base_model = resnet50(weights=weights)
# Freeze base model parameters
for param in base_model.parameters():
    param.requires_grad = False
# Replace classification head with a regression head for demand forecasting
class NPIDemandPredictor(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.features = torch.nn.Sequential(*list(base_model.children())[:-1])
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1) # Output: Predicted Year 1 Sales Volume
        )

    def forward(self, x):
        x = self.features(x)
        return self.regressor(x)
model = NPIDemandPredictor(base_model)
# Train this model on images of historical products and their total historical

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 93.6MB/s]


# 4. Probabilistic Forecasting (DeepAR)


In [5]:
!pip install gluonts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.6 MB/s eta 0:00:00


In [6]:
from gluonts.dataset.pandas import PandasDataset
from gluonts.torch.model.deepar import DeepAREstimator
from gluonts.evaluation import make_evaluation_predictions
import pandas as pd

# Prepare multiseries dataframe (index = timestamp, columns = SKUs)
dataset = PandasDataset(df, target="sales", item_id="sku")
# Initialize DeepAR Estimator
estimator = DeepAREstimator(
    freq="W", # Weekly forecasting
    prediction_length=12,
    context_length=24,
    num_layers=2,
    hidden_size=40,
    trainer_kwargs={"max_epochs": 15}
)
# Train model
predictor = estimator.train(dataset)
# Generate probabilistic forecasts
forecast_it, ts_it = make_evaluation_predictions(dataset=dataset, predictor=predictor, num_samples=100)
forecasts = list(forecast_it)
# Extract P50 (median) and P90 (high demand scenario)
print("P50:", forecasts[0].quantile(0.5))
print("P90:", forecasts[0].quantile(0.9))

# 5. Hierarchical Time Series (HTS) Reconciliation


In [7]:
import pandas as pd
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import MinTrace
from statsforecast.models import AutoARIMA
from statsforecast.core import StatsForecast

# df requires specific HTS formatting: ['ds', 'y', 'State', 'Store', 'Category', 'SKU']
# S_df is the aggregation matrix defining the hierarchy
# 1. Generate base forecasts at all levels (Top, Middle, Bottom)
models = [AutoARIMA(season_length=52)]
sf = StatsForecast(df=df, models=models, freq='W', n_jobs=-1)
Y_hat_base = sf.forecast(h=12)
# 2. Reconcile forecasts using MinTrace (Minimum Trace optimal reconciliation)
reconcilers = [MinTrace(method='mint_shrink')]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(Y_hat_df=Y_hat_base, S=S_df, tags=tags)
# Y_rec_df now contains mathematically perfectly aligned forecasts at all levels

# 6. Social Listening & Trend Ingestion via Natural Language Processing (NLP)


In [8]:
from transformers import pipeline
import pandas as pd
import numpy as np

# Initialize zero-shot classification for trend extraction
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
# Mock social media data (e.g., TikTok comments, Tweets)
social_data = [
    {"date": "2023-10-01", "text": "I absolutely need these chunky loafers right now."},
    {"date": "2023-10-02", "text": "Chunky loafers look so heavy and ugly, no thanks."}
]
df_social = pd.DataFrame(social_data)
# Score intent to purchase
labels = ["purchase intent", "dislike", "neutral"]
df_social['intent_score'] = df_social['text'].apply(
    lambda x: classifier(x, labels)['scores'][0] if classifier(x, labels)['labels'][0] == 'purchase intent' else 0
)
# Aggregate daily intent score to use as an exogenous variable in XGBoost/SARIMAX
daily_trend_signal = df_social.groupby('date')['intent_score'].sum().reset_index()

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [9]:
daily_trend_signal

,date,intent_score
0,2023-10-01,0.941965
1,2023-10-02,0.000000


# 7. Intermittent Demand Modeling (Croston’s Method)


In [11]:
import numpy as np
import pandas as pd
from sktime.forecasting.croston import Croston

# Mock intermittent data (lots of zeros)
y = pd.Series([0, 0, 5, 0, 0, 0, 3, 0, 0, 10, 0, 0])
# Initialize Croston's method with smoothing parameter alpha
# SBC (Syntetos-Boylan) is generally preferred to remove Croston bias
forecaster = Croston(smoothing=0.1)
forecaster.fit(y)
# Predict next 5 periods
y_pred = forecaster.predict(fh=[1, 2, 3, 4, 5])
print("Intermittent Forecasts:\n", y_pred)

# 8. Dynamic Lifecycle Modeling (Parametric Growth Curves)


In [12]:
import numpy as np
from scipy.optimize import curve_fit

# Bass Diffusion Equation
def bass_diffusion(t, p, q, m):
    # p = coefficient of innovation, q = coefficient of imitation, m = market potential
    return m * (((p + q)**2 / p) * np.exp(-(p + q) * t)) / (1 + (q / p) * np.exp(-(p + q) * t))**2
# Time periods (e.g., weeks 1 to 10) and actual early sales data
t_data = np.array([1, 2, 3, 4, 5])
sales_data = np.array([100, 250, 500, 800, 1100])
# Fit the curve to early data
# Bounds ensure p, q, and m stay in logical positive ranges
popt, pcov = curve_fit(bass_diffusion, t_data, sales_data, bounds=(0, [1., 1., 100000.]))
p, q, m = popt
# Forecast the entire lifecycle (Weeks 1 to 20)
t_forecast = np.arange(1, 21)
lifecycle_forecast = bass_diffusion(t_forecast, p, q, m)
print(f"Market Potential (m): {m:.0f}")
print(f"Peak Week: {np.argmax(lifecycle_forecast) + 1}")

Market Potential (m): 5673
Peak Week: 6


# 9. Causal Forecasting with Multi-Modal External Indicators


In [13]:
import pandas as pd
import statsmodels.api as sm

# Assume df has 'sales', 'avg_temp', and 'competitor_discount_depth'
endog = df['sales']
exog = df[['avg_temp', 'competitor_discount_depth']]
# Fit SARIMAX model (Seasonal ARIMA with eXogenous variables)
# Order = (AR, I, MA), Seasonal Order = (P, D, Q, s)
model = sm.tsa.statespace.SARIMAX(
    endog,
    exog=exog,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 0, 52),
    enforce_stationarity=False,
    enforce_invertibility=False
)
results = model.fit(disp=False)
# To forecast, we must provide future exogenous data (e.g., weather forecasts)
future_exog = pd.DataFrame({
    'avg_temp': [65.2, 68.1, 70.5],
    'competitor_discount_depth': [0.1, 0.15, 0.2]
})
forecast = results.get_forecast(steps=3, exog=future_exog)
print(forecast.predicted_mean)

# 10. Reinforcement Learning (RL) for Joint Forecasting and Replenishment


In [14]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from ray.rllib.algorithms.ppo import PPOConfig

class InventoryEnv(gym.Env):
    def __init__(self, env_config):
        # Action: How many units to order (0 to 100)
        self.action_space = spaces.Discrete(100)
        # State: [Current Inventory, Day of Week, Pending Orders]
        self.observation_space = spaces.Box(low=0, high=1000, shape=(3,), dtype=np.float32)
        self.inventory = 50

    def step(self, action):
        # 1. Simulate demand (this would normally tie to an external dataset/model)
        demand = np.random.poisson(20)

        # 2. Update inventory based on action and demand
        self.inventory += action - demand

        # 3. Calculate Reward (Profit minus holding costs minus stockout penalties)
        reward = 0
        if self.inventory < 0:
            reward -= abs(self.inventory) * 10 # Stockout penalty
            self.inventory = 0
        else:
            reward -= self.inventory * 0.5 # Holding cost
            reward += demand * 15 # Profit per unit

        # 4. Next state
        state = np.array([self.inventory, np.random.randint(7), action], dtype=np.float32)
        terminated = False # Endless simulation for continuous replenishment

        return state, reward, terminated, False, {}
# Configure Proximal Policy Optimization (PPO) to learn the replenishment policy
config = (
    PPOConfig()
    .environment(env=InventoryEnv, env_config={})
    .rollouts(num_rollout_workers=2)
)
algo = config.build()
# Train the agent to maximize profit over time
for _ in range(5):
    result = algo.train()
    print(f"Mean Reward: {result['env_runners']['episode_reward_mean']}")